# Feature Engineering

Transforming raw play-by-play events into machine-learning-ready features.

In [1]:
import pandas as pd
import re
import numpy as np

df = pd.read_csv("../data/pbp2024.csv")

## Converting Game Clock to Seconds

In [2]:
def convert_clock_to_seconds(clock_str):
    match = re.match(r"PT(\d+)M(\d+)", clock_str)

    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))

        return minutes * 60 + seconds

In [3]:
convert_clock_to_seconds("PT08M04.00S")

484

## Building Features for a Single Game

In [4]:
game = df['gameid'].iloc[0]

single_game = df[df['gameid'] == game]

scoring = single_game[single_game['h_pts'].notna()].copy()

scoring['score_diff'] = ( scoring['h_pts'] - scoring['a_pts'] )

scoring['quarter_seconds_remaining'] = ( scoring['clock'] .apply(convert_clock_to_seconds) )

scoring['game_seconds_remaining'] = ( (4 - scoring['period']) * 720 + scoring['quarter_seconds_remaining'] )

scoring[['score_diff', 'game_seconds_remaining']].tail()

,score_diff,game_seconds_remaining
456,23.0,83
457,21.0,74
461,23.0,24
462,21.0,17
463,21.0,0


## Creating the Target Variable

In [5]:
final_diff = scoring['score_diff'].iloc[-1]

home_win_bool = 1 if final_diff > 0 else 0

home_win_bool

1

## Building the Training Dataset

In [6]:
gameids = df['gameid'].unique()
all_games_scoring = []

for game in gameids:
    single_game = df[df['gameid'] == game]

    # Keep ALL scoring rows first (including OT)
    full_scoring = single_game[single_game['h_pts'].notna()].copy()

    if full_scoring.empty:
        continue

    # Determine actual game winner from final score
    final_diff = ( full_scoring['h_pts'].iloc[-1] - full_scoring['a_pts'].iloc[-1] )

    home_win_bool = 1 if final_diff > 0 else 0

    # Now remove OT rows for feature generation
    scoring = full_scoring[full_scoring['period'] <= 4].copy()

    scoring['score_diff'] = scoring['h_pts'] - scoring['a_pts']
    scoring['quarter_seconds_remaining'] = ( scoring['clock'].apply(convert_clock_to_seconds) )

    scoring['game_seconds_remaining'] = ((4 - scoring['period']) * 720 + scoring['quarter_seconds_remaining'])

    scoring['home_win_bool'] = home_win_bool

    all_games_scoring.append(scoring[['gameid', 'score_diff', 'game_seconds_remaining', 'home_win_bool']])


training_df = pd.concat(
    all_games_scoring,
    ignore_index=True
)

In [7]:
training_df.describe()

,gameid,score_diff,game_seconds_remaining,home_win_bool
count,1.676620e+05,167662.000000,167662.000000,167662.000000
mean,2.360416e+07,1.439915,1408.487958,0.548174
std,5.067300e+06,12.001202,833.441982,0.497675
min,2.230000e+07,-53.000000,0.000000,0.000000
25%,2.230032e+07,-6.000000,720.000000,0.000000
50%,2.230064e+07,1.000000,1440.000000,1.000000
75%,2.230098e+07,8.000000,2149.000000,1.000000
max,5.230021e+07,62.000000,2880.000000,1.000000


In [8]:
training_df['period'].value_counts().sort_index()

KeyError: 'period'

### Overtime Handling

Overtime periods create negative values for game_seconds_remaining because the original feature engineering assumes a four-quarter game.

To avoid introducing inconsistent time values, overtime rows were removed while retaining the final game result as the target variable.